# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [1]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests
import cartopy
import geoviews
import pyproj

# Import API key
from api_keys import geoapify_key

In [2]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,adamstown,-25.0660,-130.1015,21.29,87,100,7.03,PN,1728613084
1,1,ilulissat,69.2167,-51.1000,-1.99,43,46,1.03,GL,1728613084
2,2,lepsy,46.2350,78.9456,17.30,35,100,2.88,KZ,1728613085
3,3,bethel,41.3712,-73.4140,9.05,69,0,3.09,US,1728613085
4,4,black forest,39.0131,-104.7008,16.55,29,0,3.60,US,1728613085


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [3]:
# Create the map plot
map_plot = city_data_df.hvplot.points(
    'Lng',
    'Lat',
    geo=True,
    tiles='EsriImagery',
    size='Humidity',
    color='City',
    frame_width=800,
    frame_height=600
)

# Display the map
map_plot


:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [4]:
# Narrow down cities that fit criteria and drop any results with null values
min_temp_s = city_data_df["Max Temp"] > 21
max_temp_s = city_data_df["Max Temp"] < 27
wind_speed_s = city_data_df["Wind Speed"] > 5

city_temp_range_df = city_data_df[min_temp_s & max_temp_s & wind_speed_s]
city_temp_range_df


,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,adamstown,-25.0660,-130.1015,21.29,87,100,7.03,PN,1728613084
7,7,mont-dore,-22.2833,166.5833,24.07,62,0,5.80,NC,1728613086
10,10,port mathurin,-19.6833,63.4167,22.31,79,100,10.56,MU,1728613086
60,60,uturoa,-16.7333,-151.4333,23.88,85,100,9.13,PF,1728613099
69,69,west island,-12.1568,96.8225,26.99,78,20,7.72,CC,1728613101
99,99,bandarbeyla,9.4942,50.8122,25.16,91,64,7.22,SO,1728613109
117,117,afaahiti,-17.7500,-149.2833,21.69,90,100,9.62,PF,1728613113
145,145,piacabucu,-10.4056,-36.4344,25.43,69,100,5.38,BR,1728613120
150,150,ta`u,-14.2336,-169.5144,26.72,81,51,8.61,AS,1728613121
154,154,nova sintra,14.8667,-24.7167,26.64,85,100,9.93,CV,1728613121


In [5]:
# Drop any rows with null values
clean_city_temp_range_df = city_temp_range_df.dropna()

# Display sample data
clean_city_temp_range_df

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,adamstown,-25.0660,-130.1015,21.29,87,100,7.03,PN,1728613084
7,7,mont-dore,-22.2833,166.5833,24.07,62,0,5.80,NC,1728613086
10,10,port mathurin,-19.6833,63.4167,22.31,79,100,10.56,MU,1728613086
60,60,uturoa,-16.7333,-151.4333,23.88,85,100,9.13,PF,1728613099
69,69,west island,-12.1568,96.8225,26.99,78,20,7.72,CC,1728613101
99,99,bandarbeyla,9.4942,50.8122,25.16,91,64,7.22,SO,1728613109
117,117,afaahiti,-17.7500,-149.2833,21.69,90,100,9.62,PF,1728613113
145,145,piacabucu,-10.4056,-36.4344,25.43,69,100,5.38,BR,1728613120
150,150,ta`u,-14.2336,-169.5144,26.72,81,51,8.61,AS,1728613121
154,154,nova sintra,14.8667,-24.7167,26.64,85,100,9.93,CV,1728613121


### Step 3: Create a new DataFrame called `hotel_df`.

In [6]:
# Use the Pandas copy function to create DataFrame called hotel_df to store the city, country, coordinates, and humidity
hotel_df = clean_city_temp_range_df.copy()

# Add an empty column, "Hotel Name," to the DataFrame so you can store the hotel found using the Geoapify API
hotel_df["Hotel Name"] = ""

# Display sample data
hotel_df

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date,Hotel Name
0,0,adamstown,-25.0660,-130.1015,21.29,87,100,7.03,PN,1728613084,
7,7,mont-dore,-22.2833,166.5833,24.07,62,0,5.80,NC,1728613086,
10,10,port mathurin,-19.6833,63.4167,22.31,79,100,10.56,MU,1728613086,
60,60,uturoa,-16.7333,-151.4333,23.88,85,100,9.13,PF,1728613099,
69,69,west island,-12.1568,96.8225,26.99,78,20,7.72,CC,1728613101,
99,99,bandarbeyla,9.4942,50.8122,25.16,91,64,7.22,SO,1728613109,
117,117,afaahiti,-17.7500,-149.2833,21.69,90,100,9.62,PF,1728613113,
145,145,piacabucu,-10.4056,-36.4344,25.43,69,100,5.38,BR,1728613120,
150,150,ta`u,-14.2336,-169.5144,26.72,81,51,8.61,AS,1728613121,
154,154,nova sintra,14.8667,-24.7167,26.64,85,100,9.93,CV,1728613121,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [7]:
# Set parameters to search for a hotel
radius = 10000
limit = 1
categories = "accommodation.hotel"

#create a params dictionary
params = {
    "categories":categories,
    "limit":limit,
    "apiKey":geoapify_key     
}

# Print a message to follow up the hotel search
print("Starting hotel search")

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # get latitude, longitude from the DataFrame
    lng = hotel_df.loc[index, "Lng"]
    lat = hotel_df.loc[index, "Lat"]
    # Add filter and bias parameters with the current city's latitude and longitude to the params dictionary
    params["filter"] = f"circle:{lng},{lat},{radius}"
    params["bias"] = f"proximity:{lng},{lat}"
    
    # Set base URL
    base_url = "https://api.geoapify.com/v2/places"


    # Make and API request using the params dictionaty
    name_address = requests.get(base_url, params = params)
    print(name_address.url)
    # Convert the API response to JSON format
    name_address =  name_address = name_address.json()
    
    # Grab the first hotel from the results and store the name in the hotel_df DataFrame
    try:
        hotel_df.loc[index, "Hotel Name"] = name_address["features"][0]["properties"]["name"]
    except (KeyError, IndexError):
        # If no hotel is found, set the hotel name as "No hotel found".
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"
        
    # Log the search results
    print(f"{hotel_df.loc[index, 'City']} - nearest hotel: {hotel_df.loc[index, 'Hotel Name']}")

# Display sample data
hotel_df

Starting hotel search
https://api.geoapify.com/v2/places?categories=accommodation.hotel&limit=1&apiKey=2746d74c361648528c3ad5d86cbb52c0&filter=circle%3A-130.1015%2C-25.066%2C10000&bias=proximity%3A-130.1015%2C-25.066
adamstown - nearest hotel: No hotel found
https://api.geoapify.com/v2/places?categories=accommodation.hotel&limit=1&apiKey=2746d74c361648528c3ad5d86cbb52c0&filter=circle%3A166.5833%2C-22.2833%2C10000&bias=proximity%3A166.5833%2C-22.2833
mont-dore - nearest hotel: Les Cases de Plum
https://api.geoapify.com/v2/places?categories=accommodation.hotel&limit=1&apiKey=2746d74c361648528c3ad5d86cbb52c0&filter=circle%3A63.4167%2C-19.6833%2C10000&bias=proximity%3A63.4167%2C-19.6833
port mathurin - nearest hotel: Escale Vacances
https://api.geoapify.com/v2/places?categories=accommodation.hotel&limit=1&apiKey=2746d74c361648528c3ad5d86cbb52c0&filter=circle%3A-151.4333%2C-16.7333%2C10000&bias=proximity%3A-151.4333%2C-16.7333
uturoa - nearest hotel: Hawaiki Nui hotel
https://api.geoapify.c

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date,Hotel Name
0,0,adamstown,-25.0660,-130.1015,21.29,87,100,7.03,PN,1728613084,No hotel found
7,7,mont-dore,-22.2833,166.5833,24.07,62,0,5.80,NC,1728613086,Les Cases de Plum
10,10,port mathurin,-19.6833,63.4167,22.31,79,100,10.56,MU,1728613086,Escale Vacances
60,60,uturoa,-16.7333,-151.4333,23.88,85,100,9.13,PF,1728613099,Hawaiki Nui hotel
69,69,west island,-12.1568,96.8225,26.99,78,20,7.72,CC,1728613101,Cocos Village Bungalows
99,99,bandarbeyla,9.4942,50.8122,25.16,91,64,7.22,SO,1728613109,No hotel found
117,117,afaahiti,-17.7500,-149.2833,21.69,90,100,9.62,PF,1728613113,Omati Lodge
145,145,piacabucu,-10.4056,-36.4344,25.43,69,100,5.38,BR,1728613120,O Leão
150,150,ta`u,-14.2336,-169.5144,26.72,81,51,8.61,AS,1728613121,No hotel found
154,154,nova sintra,14.8667,-24.7167,26.64,85,100,9.93,CV,1728613121,Residência Ka Dencho


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [8]:
%%capture --no-display

# Configure the map plot
hotel_map_plot = hotel_df.hvplot.points(
    "Lng",
    "Lat",
    geo = True,
    tiles = "EsriImagery",
    frame_width = 800,
    frame_height = 600, 
    scale = 0.5,
    color = "City"
)

# Display the map
hotel_map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City)